# Dana Engine — Collaboration Notebook

> **Platform:** Google Colab free tier — **T4 GPU (16 GB VRAM)**  
> **Purpose:** Test, profile, and extend the Dana MoE inference engine.  
> **Session limit:** ~3–12 h. Models download to Colab's local disk (`/content/dana/`) — **no Google Drive required**.

---

### ⚠ Before running anything

1. **Runtime → Change runtime type → T4 GPU** (top-right Colab menu)  
2. **Run Section 0 top-to-bottom** every new session (installs packages + downloads model if needed)  
3. GPU cells auto-skip if no CUDA — safe to run the whole notebook on CPU too

---

## Table of Contents

| # | Section | GPU required? |
|---|---------|---------------|
| 0 | [Setup & Installation](#0-setup) | No |
| 1 | [Project Overview](#1-overview) | No |
| 2 | [CPU Smoke Tests (all packages)](#2-cpu-tests) | No |
| 3 | [Integration Benchmark (CPU)](#3-integration) | No |
| 4 | [GPU TODO 1 — PCIe Transfer Benchmark](#4-pcie) | **Yes** |
| 5 | [GPU TODO 2 — Tiered Tensor Store (GPU mode)](#5-tiered) | **Yes** |
| 6 | [GPU TODO 3 — Real MoE Model (Qwen)](#6-qwen) | **Yes** |
| 7 | [GPU TODO 4 — Quantization Speed Benchmark](#7-quant) | **Yes** |
| 8 | [GPU TODO 5 — Baseline vs Full Pipeline Benchmark](#8-pipeline) | **Yes** |
| 9 | [GPU TODO 6 — Expert Activation Heatmap](#9-heatmap) | **Yes** |
| 10 | [GPU TODO 7 — CUDA Kernel: Tree Verification](#10-cuda) | **Yes** |
| 11 | [GPU TODO 8 — Production API Stress Test](#11-stress) | **Yes** |
| 12 | [GPU TODO 9 — GPU Memory Profiling](#12-mem) | **Yes** |
| 13 | [Next Steps Tracker](#13-next-steps) | No |
| 14 | [Known Bugs & Quick Wins](#14-bugs) | No |

---
## 0. Setup & Installation <a id="0-setup"></a>

Run **all cells in this section** at the start of every Colab session.

### 0a. Anti-idle keepalive (prevents 90-min disconnect)

Paste this once into your browser console (`F12 → Console`) after the notebook loads:

```js
// Browser console — keeps Colab alive by clicking "Connect" every 60s
setInterval(() => {
  document.querySelector("colab-connect-button")?.shadowRoot
    ?.querySelector("paper-icon-button")?.click();
}, 60000);
```

### 0b. Set local cache directory

No Google Drive needed. Models are stored in `/content/dana/` on Colab's ephemeral disk (~78 GB free).  
**Re-download is required each new session** (~30–60 min for MoE models, ~5 min for Qwen3-8B).

In [ ]:
import os, pathlib

IN_COLAB = "COLAB_JUPYTER_TOKEN" in os.environ

# All model weights and artifacts go to Colab's local disk.
# No Google Drive access required.
# Trade-off: re-download each session (~30-60 min for MoE, ~5 min for 8B).
LOCAL_ROOT = pathlib.Path("/content/dana")
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

# Alias used by downstream cells
DRIVE_ROOT = LOCAL_ROOT

print(f"Cache dir: {LOCAL_ROOT}  ({'Colab' if IN_COLAB else 'local'} mode)")
disk = os.statvfs("/content")
free_gb = disk.f_bavail * disk.f_frsize / 1024**3
print(f"Free disk: {free_gb:.1f} GB")

In [ ]:
import os, pathlib

# --- edit these if needed ---
REPO_URL   = "https://github.com/SMKeramati/dana.git"  # replace with actual repo URL
LOCAL_ROOT = pathlib.Path("/content/dana") if IN_COLAB else pathlib.Path("/home/user/dana")
ENGINE_DIR = LOCAL_ROOT / "engine"
# ----------------------------

if not LOCAL_ROOT.exists():
    !git clone {REPO_URL} {LOCAL_ROOT}
else:
    print(f"Repo already at {LOCAL_ROOT} — pulling latest")
    !git -C {LOCAL_ROOT} pull

os.chdir(ENGINE_DIR)
print("CWD:", os.getcwd())

### 0c. Install all engine packages

In [ ]:
# Colab already ships PyTorch with CUDA.
# Only reinstall if you need a specific version.
import torch
print(f"PyTorch {torch.__version__}  CUDA available: {torch.cuda.is_available()}")
# Uncomment only if torch is missing:
# !pip install torch --index-url https://download.pytorch.org/whl/cu124 --quiet

# Install all Dana engine packages in editable mode
PKGS = [
    "tiered-tensor-store",
    "expert-cache",
    "moe-quant",
    "moe-router-predict",
    "spec-decode-tree",
    "moe-self-draft",
    "dana-engine",
]

for pkg in PKGS:
    !pip install -e {ENGINE_DIR}/{pkg}[test] -q
    print(f"✓ {pkg}")

print("\nAll packages installed.")

In [ ]:
import torch

CUDA   = torch.cuda.is_available()
DEVICE = "cuda" if CUDA else "cpu"

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {CUDA}")
if CUDA:
    props = torch.cuda.get_device_properties(0)
    vram  = props.total_memory / 1024**3
    print(f"GPU     : {props.name}")
    print(f"VRAM    : {vram:.1f} GB")
    if vram < 14:
        print("⚠ Less than 14 GB — use Qwen3-30B-A3B (INT4, ~15 GB) as fallback or reduce quant precision")
    elif vram < 20:
        print("✓ 16 GB T4 — Qwen3.5-35B-A3B fits with INT4 quantization (~17-18 GB, device_map='auto' spills a few layers to CPU RAM)")
    else:
        print("✓ Large VRAM — Qwen3.5-35B-A3B fits in BF16; or try Qwen3.5-397B-A17B with INT4")
else:
    print("No GPU — GPU cells will be skipped automatically")

---
## 1. Project Overview <a id="1-overview"></a>

Dana is a **custom MoE inference engine** built around a single thesis:

> MoE models activate only 5–15 % of parameters per token.  
> Current engines treat them like dense models — Dana does not.

### Architecture: 7 Products

| # | Package | Strategy | Key problem solved |
|---|---------|----------|--------------------|
| 1 | `moe-router-predict` | Full OSS (MIT) | Predict future experts → async prefetch |
| 2 | `expert-cache` | Full OSS (MIT) | Frequency/predictive cache (LRU→90 %+ hit) |
| 3 | `tiered-tensor-store` | Full OSS (MIT) | VRAM/RAM/SSD tensor hierarchy |
| 4 | `moe-quant` | Full OSS (MIT) | Per-expert Q2/Q4/Q8 quantization |
| 5 | `spec-decode-tree` | Limited OSS | Tree speculative decoding |
| 6 | `moe-self-draft` | Limited OSS | Self-draft speculative decode |
| 7 | `dana-engine` | **Private** | Integration layer + batching moat |

### Target hardware
Run **Qwen3.5-397B-A17B** (397B / 17B active, 512 experts) or **DeepSeek-V3.2** (685B / 37B active) at **30+ tokens/sec** on a \$20 K workstation (2× RTX 4090 + 512 GB DDR5 RAM) instead of \$200 K+ in H100s.

### Six Pillars

```
Pillar 1 — Predictive Expert Prefetching      (moe-router-predict)
Pillar 2 — Intelligent Expert Caching         (expert-cache)
Pillar 3 — Tiered Tensor Storage              (tiered-tensor-store)
Pillar 4 — Per-Expert Quantization            (moe-quant)
Pillar 5 — Tree Speculative Decoding          (spec-decode-tree)
Pillar 6 — MoE Self-Draft Speculative Decode  (moe-self-draft)
```

---
## 2. CPU Smoke Tests (all packages) <a id="2-cpu-tests"></a>

These tests run on CPU with the tiny synthetic model. All must pass before GPU work begins.

In [ ]:
import subprocess, sys

def run_tests(pkg_dir: str, verbose: bool = True) -> bool:
    """Run pytest for a package, return True on success."""
    result = subprocess.run(
        [sys.executable, "-m", "pytest", "tests/", "-v", "--tb=short", "-q"],
        cwd=ENGINE_DIR / pkg_dir,
        capture_output=not verbose,
        text=True,
    )
    if result.returncode != 0 and not verbose:
        print(result.stdout[-2000:])  # last 2KB on failure
        print(result.stderr[-500:])
    return result.returncode == 0

packages = [
    "tiered-tensor-store",
    "expert-cache",
    "moe-quant",
    "moe-router-predict",
    "spec-decode-tree",
    "moe-self-draft",
    "dana-engine",
]

results = {}
for pkg in packages:
    ok = run_tests(pkg, verbose=False)
    results[pkg] = "✓ PASS" if ok else "✗ FAIL"
    print(f"  {results[pkg]}  {pkg}")

print("\n" + "=" * 40)
all_pass = all(v.startswith("✓") for v in results.values())
print("ALL PASS" if all_pass else "SOME FAILURES — see output above")

### 2a. Explore individual package APIs

In [ ]:
# --- tiered-tensor-store ---
from tiered_tensor_store import TieredTensorStore
import torch, tempfile, pathlib

with tempfile.TemporaryDirectory() as tmp:
    store = TieredTensorStore(base_dir=tmp)

    # Store a fake expert weight in RAM tier
    w = torch.randn(64, 64)
    store.store("expert_0", w, tier="ram")

    # Load it back (auto-promotes to hot)
    w2 = store.load("expert_0")
    assert torch.allclose(w, w2), "Round-trip mismatch!"

    # SSD tier
    store.store("expert_1", torch.randn(64, 64), tier="ssd")
    w3 = store.load("expert_1")

    stats = store.stats()
    print("TieredTensorStore stats:", stats)

In [ ]:
# --- expert-cache ---
from expert_cache import FrequencyExpertCache, PredictiveExpertCache
import torch

cache = FrequencyExpertCache(capacity=4, window_size=100)

# Simulate access pattern: expert 0 and 1 are "hot"
for i in range(20):
    for eid in [0, 1]:  # hot experts
        cache.put(eid, torch.randn(32, 32))
    cache.put(i % 8 + 2, torch.randn(32, 32))  # rotating cold experts

analytics = cache.analytics()
print("Cache analytics:", analytics)

# Predictive cache with hints
pcache = PredictiveExpertCache(capacity=4)
pcache.hint([3, 5, 7])                        # pre-load hints
pcache.put(3, torch.randn(32, 32))
hit = pcache.get(3)                            # should be a hit
print(f"Predictive cache get(3): {'hit' if hit is not None else 'miss'}")

In [ ]:
# --- moe-quant ---
from moe_quant import quantize, dequantize
from moe_quant.sensitivity import SensitivityAnalyzer
from moe_quant.tier_assigner import TierAssigner
import torch

weight = torch.randn(256, 256)

for bits in [2, 4, 8]:
    qt = quantize(weight, bits=bits)
    restored = dequantize(qt)
    err = (weight - restored).abs().mean().item()
    ratio = weight.element_size() * weight.numel() / qt.nbytes
    print(f"Q{bits}: compression={ratio:.1f}x  mean_abs_err={err:.4f}")

In [ ]:
# --- moe-router-predict ---
from dana_engine.model.config import TinyMoEConfig
from dana_engine.model.transformer import TinyMoETransformer
from moe_router_predict.predictor import RouterPredictor
from moe_router_predict.residency import ExpertResidencyTracker
import torch

cfg = TinyMoEConfig.micro()
model = TinyMoETransformer(cfg).eval()

predictor = RouterPredictor(model, num_steps=3)
input_ids = torch.randint(0, cfg.vocab_size, (1, 8))

with torch.no_grad():
    out = model(input_ids)
    hidden = out.all_hidden_states[-1]

predictions = predictor.predict(hidden, num_steps=3)
print(f"Predicted {len(predictions)} steps ahead")
for i, pred in enumerate(predictions):
    print(f"  step+{i+1}: experts {pred.expert_ids}  confidence={pred.confidence:.2f}")

In [ ]:
# --- spec-decode-tree ---
from spec_decode_tree.tree_spec import TreeSpecDecoder
from spec_decode_tree.acceptance import SpeculativeAcceptor
import torch

cfg = TinyMoEConfig.micro()
model = TinyMoETransformer(cfg).eval()

decoder = TreeSpecDecoder(model, depth=2, width=2)
input_ids = torch.randint(0, cfg.vocab_size, (1, 8))

with torch.no_grad():
    tree_result = decoder.generate_tree(input_ids)

print(f"Tree paths generated : {tree_result.num_paths}")
print(f"Tree depth           : {tree_result.depth}")

In [ ]:
# --- moe-self-draft ---
from moe_self_draft.self_draft import MoeSelfDrafter
from moe_self_draft.verify import SelfDraftVerifier
import torch

cfg = TinyMoEConfig.micro()
model = TinyMoETransformer(cfg).eval()

drafter  = MoeSelfDrafter(model, num_active_override=1)
verifier = SelfDraftVerifier(model)

input_ids = torch.randint(0, cfg.vocab_size, (1, 8))

with torch.no_grad():
    draft_result = drafter.draft(input_ids, num_draft_tokens=4)
    accepted = verifier.verify(input_ids, draft_result)

print(f"Drafted  : {len(draft_result.token_ids)} tokens")
print(f"Accepted : {len(accepted)} tokens")
print(f"Accept rate: {len(accepted) / len(draft_result.token_ids):.0%}")

---
## 3. Integration Benchmark (CPU) <a id="3-integration"></a>

Runs the full `test_integration_benchmark.py` suite and displays the summary table.

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "-m", "pytest",
     "dana-engine/tests/test_integration_benchmark.py",
     "-v", "-s", "--tb=short"],
    cwd=ENGINE_DIR,
    capture_output=False,
    text=True,
)
print("Exit code:", result.returncode)

### 3a. Inline benchmark (no pytest overhead)

In [ ]:
import time, statistics, torch
from dana_engine.model.config import TinyMoEConfig
from dana_engine.model.transformer import TinyMoETransformer
from dana_engine.naive_inference import greedy_generate
from dana_engine.pipeline import DanaInferencePipeline, PipelineConfig
from dana_engine.speculative.self_draft_runner import OptimizedSelfDraftRunner, SelfDraftRunnerConfig
from dana_engine.speculative.tree_runner import OptimizedTreeRunner, TreeRunnerConfig

cfg = TinyMoEConfig.micro()
model = TinyMoETransformer(cfg).eval()
MAX_NEW = 32
N_RUNS  = 5

def make_input():
    return torch.randint(0, cfg.vocab_size, (1, 8))

def bench(fn, label):
    tps_list = []
    for _ in range(N_RUNS):
        r = fn()
        tps_list.append(r)
    m = statistics.median(tps_list)
    print(f"  {label:<45} {m:>8.1f}  TPS")
    return m

print("\nDANA ENGINE — CPU BENCHMARK")
print("=" * 60)

# 1. Naive
def naive():
    ids = make_input()
    t0 = time.perf_counter()
    r = greedy_generate(model, ids, max_new_tokens=MAX_NEW)
    elapsed = time.perf_counter() - t0
    new = r.tokens[0, ids.shape[1]:].tolist()
    return len(new) / elapsed

naive_tps = bench(naive, "Naïve greedy")

# 2. Self-draft spec decode
runner = OptimizedSelfDraftRunner(model, SelfDraftRunnerConfig(num_draft_tokens=4, max_new_tokens=MAX_NEW))
def self_draft():
    ids = make_input()
    _, meta = runner.run(ids, max_new_tokens=MAX_NEW)
    return meta["tokens_per_second"]

sd_tps = bench(self_draft, "Self-draft spec decode (k=4)")

# 3. Tree spec decode
tree_runner = OptimizedTreeRunner(model, TreeRunnerConfig(initial_depth=2, initial_width=2, max_new_tokens=MAX_NEW))
def tree_decode():
    ids = make_input()
    _, meta = tree_runner.run(ids, max_new_tokens=MAX_NEW)
    return meta["tokens_per_second"]

tree_tps = bench(tree_decode, "Tree spec decode (d=2,w=2)")

# 4. Full pipeline
pipeline = DanaInferencePipeline(PipelineConfig(
    model_config=cfg,
    enable_spec_decode=True,
    num_draft_tokens=4,
    enable_prefetch=True,
    max_new_tokens=MAX_NEW,
))
def full_pipeline():
    ids = make_input()
    r = pipeline.generate(ids, max_new_tokens=MAX_NEW)
    return r.tokens_per_second

full_tps = bench(full_pipeline, "Full pipeline (spec+prefetch)")

print("=" * 60)
print(f"  Spec/Naïve speedup: {full_tps / naive_tps:.2f}x")

---
## 4. GPU TODO 1 — PCIe Transfer Benchmark <a id="4-pcie"></a>

**Why:** PCIe is the core bottleneck the engine solves. Measure real RAM→VRAM latency.

**T4 bandwidth:** PCIe 3.0 × 16 → ~12–14 GB/s practical (vs 4090 PCIe 4.0 ~28 GB/s).  
**Expert weight sizes:** Qwen3.5-35B-A3B has 256 experts per layer, each ~4–12 MB in FP16 (hidden=2048, intermediate≈1024 per expert). At INT4 (working format), each expert is ~2–6 MB — so loading a cold expert from CPU RAM to VRAM costs ~0.15–0.5 ms at T4 PCIe bandwidth.

In [ ]:
import torch, time, statistics

if not torch.cuda.is_available():
    print("No GPU — skipping PCIe benchmark")
else:
    # Qwen3.5-35B-A3B expert weight shapes (approximate):
    #   gate_proj / up_proj: (1024, 2048) FP16 = ~4 MB per matrix
    #   full expert (3 projections stacked): ~12 MB FP16 | ~6 MB INT4
    sizes = [
        ("small  512×512   FP16  (~0.5 MB)",  512,  512,  torch.float16),
        ("medium 2048×1024 FP16  (~4 MB)",   2048, 1024,  torch.float16),   # single Qwen3.5-35B expert proj
        ("large  2048×3072 FP16  (~12 MB)",  2048, 3072,  torch.float16),   # full Qwen3.5-35B expert (3 proj)
    ]

    print(f"PCIe Transfer Benchmark — {torch.cuda.get_device_name(0)}\n")
    print(f"  {'Tensor':<44} {'Median ms':>10}  {'GB/s':>6}")
    print("  " + "-" * 65)

    for label, rows, cols, dtype in sizes:
        t = torch.randn(rows, cols, dtype=dtype).pin_memory()
        size_bytes = t.element_size() * t.numel()

        # Warm up
        for _ in range(3):
            t.cuda(non_blocking=False); torch.cuda.synchronize()

        times = []
        for _ in range(50):
            t0 = time.perf_counter()
            t.cuda(non_blocking=False)
            torch.cuda.synchronize()
            times.append(time.perf_counter() - t0)

        med_ms  = statistics.median(times) * 1000
        bw_gbps = size_bytes / (statistics.median(times) * 1e9)
        print(f"  {label:<44} {med_ms:>10.2f}  {bw_gbps:>6.1f}")

    print("\nT4 PCIe 3.0×16 practical: ~12–14 GB/s")
    print("Each Qwen3.5-35B-A3B expert (~4 MB FP16) costs ~0.3 ms to load cold from CPU RAM")

---
## 5. GPU TODO 2 — Tiered Tensor Store (GPU mode) <a id="5-tiered"></a>

**Status:** Hot tier currently stores CPU tensors. This section validates GPU promotion.  
**Fix needed:** `tier_manager.py` hot tier should call `.cuda()` when CUDA is available.

**T4 budget:** Use `hot_budget_bytes = 12 * 1024**3` (12 GB) — leave 4 GB for model activations.

In [ ]:
import torch, tempfile
from tiered_tensor_store import TieredTensorStore

if not torch.cuda.is_available():
    print("No GPU — skipping GPU tier test")
else:
    with tempfile.TemporaryDirectory() as tmp:
        # T4: 16 GB total — use 12 GB for hot tier, leave 4 GB for activations + model non-expert weights
        store = TieredTensorStore(
            base_dir=tmp,
            hot_budget_bytes=12 * 1024**3,
        )

        # Simulate a real Qwen3.5-35B-A3B expert weight (gate_proj: 2048×1024 FP16 = ~4 MB per expert)
        # 256 experts per layer × ~4 MB = ~1 GB per layer; hot tier caches the most-used subset
        expert = torch.randn(2048, 1024, dtype=torch.float16)
        size_mb = expert.element_size() * expert.numel() / 1024**2
        print(f"Expert tensor: {expert.shape}  {size_mb:.1f} MB FP16  (single gate_proj, Qwen3.5-35B-A3B)")

        store.store("expert_0", expert, tier="ram")
        loaded = store.load("expert_0")
        is_cuda = loaded.device.type == "cuda"

        if is_cuda:
            used_mb = torch.cuda.memory_allocated() / 1024**2
            print(f"✓ Expert promoted to CUDA — VRAM used: {used_mb:.1f} MB")
        else:
            print("⚠ Expert still on CPU — hot tier GPU promotion not yet wired")
            print("  Fix (P2a): in tier_manager.py replace")
            print("    self._hot[key] = tensor")
            print("  with")
            print("    self._hot[key] = tensor.cuda() if torch.cuda.is_available() else tensor")

        print("Store stats:", store.stats())

---
## 6. GPU TODO 3 — Real MoE Model (Qwen) <a id="6-qwen"></a>

### Does Colab have RAM and VRAM simultaneously? YES — with caveats.

A single Colab runtime has:
- **VRAM**: 15 GB (T4)
- **CPU RAM**: ~12 GB (free tier) / ~25 GB (Pro) / ~52 GB (Pro+)
- Both are accessible to `device_map="auto"` in a single Python process

`device_map="auto"` places **hot layers** (attention, routing, embedding) in VRAM and **cold layers** (coldest expert FFNs) in CPU RAM — exactly what Dana is designed to accelerate.

**Why the process dies without memory caps:**  
bitsandbytes loads BF16 shards from disk, holds a chunk in CPU RAM, quantizes to NF4, then places the result. Without explicit caps, it grabs memory greedily. When it hits either ceiling, the **OS OOM-killer terminates the entire kernel** — not a Python exception, a full runtime crash with no recovery.

**Fix:** `max_memory={0: "11GiB", "cpu": "8GiB"}` — hard caps that accelerate respects when distributing layers.

### Model selection (auto-selected by preflight cell)

| Model | Total | Active | Experts | NF4 size | Required budget |
|---|---|---|---|---|---|
| `Qwen3.5-35B-A3B` | 35B | 3B | 256 | ~18 GB | ≥ 19.5 GB |
| `Qwen3-30B-A3B` | 30B | 3B | 128 | ~15.5 GB | ≥ 16.5 GB |
| `Qwen3-8B` (dense fallback) | 8B | 8B | — | ~4.5 GB | any |

Standard Colab (12 GB RAM + 15 GB VRAM = ~15.5 GB safe budget after headroom) typically selects **Qwen3-30B-A3B**. Colab Pro/Pro+ gets the 35B.

**Model downloads to `/content/dana/` on Colab's local disk** (~78 GB free, no Drive needed).  
Re-download is required each new session — plan to run all tests in one go.

In [ ]:
!pip install transformers huggingface_hub accelerate bitsandbytes psutil --quiet

import psutil, torch

# ── Memory preflight ───────────────────────────────────────────────────────────
# Colab gives us VRAM and CPU RAM simultaneously in a single runtime.
# device_map="auto" distributes layers across both, but WITHOUT explicit caps
# the loader will exceed one ceiling and the OS kills the whole kernel (no recovery).
# We set hard caps with headroom, then pick the largest model that fits safely.
# ──────────────────────────────────────────────────────────────────────────────

has_gpu = torch.cuda.is_available()
if has_gpu:
    gpu_prop   = torch.cuda.get_device_properties(0)
    gpu_total  = gpu_prop.total_memory / 1024**3
    gpu_free   = (gpu_prop.total_memory - torch.cuda.memory_allocated()) / 1024**3
else:
    gpu_total  = gpu_free = 0.0

ram_total = psutil.virtual_memory().total     / 1024**3
ram_free  = psutil.virtual_memory().available / 1024**3

# ── Headroom rules ────────────────────────────────────────────────────────────
#   VRAM: reserve 3.5 GB for CUDA context, activations, KV cache
#   RAM : reserve 4.0 GB for OS, Python heap, bitsandbytes loading spikes
#   (loading BF16 → NF4 temporarily holds a shard in CPU RAM — the spike is real)
VRAM_CAP_GB = max(0.0, gpu_free  - 3.5)
RAM_CAP_GB  = max(0.0, ram_free  - 4.0)
budget_gb   = VRAM_CAP_GB + RAM_CAP_GB

print(f"GPU VRAM   : {gpu_free:.1f} GB free / {gpu_total:.1f} GB total")
print(f"CPU RAM    : {ram_free:.1f} GB free / {ram_total:.1f} GB total")
print(f"Safe budget: {VRAM_CAP_GB:.1f} GB (VRAM cap)  +  {RAM_CAP_GB:.1f} GB (RAM cap)  =  {budget_gb:.1f} GB\n")

# ── Model selection ───────────────────────────────────────────────────────────
#  NF4 footprint (approximate):  35B ~18 GB  |  30B ~15.5 GB  |  8B ~4.5 GB
#  Add ~1.5 GB loading overhead on top when deciding thresholds.
if not has_gpu:
    MODEL_ID = None
    print("No GPU detected — skipping model selection (run in a GPU runtime)")
elif budget_gb >= 19.5:
    MODEL_ID   = "Qwen/Qwen3.5-35B-A3B"    # 35B total | 3B active | 256 experts
    model_note = "NF4 ~18 GB — fits with your memory budget"
elif budget_gb >= 16.5:
    MODEL_ID   = "Qwen/Qwen3-30B-A3B"       # 30B total | 3B active | 128 experts
    model_note = "NF4 ~15.5 GB — safer fallback for standard Colab RAM"
else:
    MODEL_ID   = "Qwen/Qwen3-8B"            # 8B dense  | still shows quant + adapter patterns
    model_note = "NF4 ~4.5 GB — low-RAM fallback; dense but all adapter code still applies"
    print("⚠  Budget < 16.5 GB — falling back to 8B dense model.")
    print("   For MoE models consider Colab Pro (25 GB RAM) or Pro+ (52 GB RAM).\n")

if MODEL_ID:
    print(f"→ Selected: {MODEL_ID}")
    print(f"   {model_note}")

# ── Memory caps string for HuggingFace ────────────────────────────────────────
# Floor to int so we never accidentally exceed the cap
VRAM_CAP_STR = f"{max(1, int(VRAM_CAP_GB))}GiB"
RAM_CAP_STR  = f"{max(1, int(RAM_CAP_GB))}GiB"
MAX_MEMORY   = {0: VRAM_CAP_STR, "cpu": RAM_CAP_STR}
print(f"\nmax_memory caps → GPU: {VRAM_CAP_STR}  |  CPU: {RAM_CAP_STR}")

In [ ]:
import pathlib
from huggingface_hub import snapshot_download

if MODEL_ID is None:
    print("No GPU — skipping download")
else:
    LOCAL_DIR = str(DRIVE_ROOT / MODEL_ID.replace("/", "--"))

    if pathlib.Path(LOCAL_DIR).exists() and any(pathlib.Path(LOCAL_DIR).iterdir()):
        print(f"Model already cached at {LOCAL_DIR} — skipping download")
    else:
        sizes = {"Qwen/Qwen3.5-35B-A3B": "~70 GB", "Qwen/Qwen3-30B-A3B": "~60 GB", "Qwen/Qwen3-8B": "~16 GB"}
        est   = sizes.get(MODEL_ID, "unknown size")
        print(f"Downloading {MODEL_ID} → {LOCAL_DIR}")
        print(f"BF16 safetensors are {est} — expect 15–60 min on Colab's network.")
        print("Note: downloads to local Colab disk, re-download required each new session.")
        snapshot_download(MODEL_ID, local_dir=LOCAL_DIR)
        print("Download complete.")

In [ ]:
import torch, time, gc

if MODEL_ID is None:
    print("No GPU — skipping")
else:
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    # ── Free whatever is already on GPU before loading ─────────────────────────
    gc.collect()
    torch.cuda.empty_cache()

    # ── INT4/NF4 quantization config ───────────────────────────────────────────
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

    # ── max_memory: hard caps computed in preflight cell ──────────────────────
    # device_map="auto" respects these caps and distributes layers accordingly:
    #   hot layers (attention, routing) → VRAM
    #   cold expert FFN layers          → CPU RAM (slow, but doesn't crash)
    # Without these caps, the loader grabs all available memory and the OS
    # OOM-kills the kernel with no recovery.
    print(f"Loading {MODEL_ID} in NF4 ...")
    print(f"  max_memory = {MAX_MEMORY}")
    print(f"  (loading BF16→NF4 can take 3–8 min on T4)")

    try:
        tokenizer = AutoTokenizer.from_pretrained(LOCAL_DIR)
        model_hf  = AutoModelForCausalLM.from_pretrained(
            LOCAL_DIR,
            quantization_config=bnb_config,
            device_map="auto",
            max_memory=MAX_MEMORY,      # ← explicit caps: never hits either ceiling
            low_cpu_mem_usage=True,     # ← streams shards; reduces loading RAM spike
        )
        model_hf.eval()
    except torch.cuda.OutOfMemoryError:
        print("CUDA OOM during load — try restarting runtime and reducing MAX_NEW_TOKENS")
        raise
    except MemoryError:
        print("CPU RAM OOM during load — try Colab Pro or a smaller model")
        raise

    # ── Memory report ──────────────────────────────────────────────────────────
    vram_used = torch.cuda.memory_allocated() / 1024**3
    vram_res  = torch.cuda.memory_reserved()  / 1024**3
    import psutil
    ram_used  = (psutil.virtual_memory().total - psutil.virtual_memory().available) / 1024**3
    total_p   = sum(p.numel() for p in model_hf.parameters()) / 1e9
    print(f"\nLoaded {total_p:.1f}B params")
    print(f"  VRAM allocated : {vram_used:.2f} GB  (reserved: {vram_res:.2f} GB)")
    print(f"  CPU RAM in use : {ram_used:.1f} GB")

    # ── Quick sanity generation ────────────────────────────────────────────────
    # Keep max_new_tokens small: each token grows the KV cache by
    #   2 * num_layers * num_heads * head_dim * batch_size bytes
    # On 35B that's ~0.5 MB/token → 64 tokens ≈ 32 MB (safe)
    MAX_NEW_TOKENS = 48
    prompt  = "The key insight about Mixture-of-Experts inference is"
    inputs  = tokenizer(prompt, return_tensors="pt").to("cuda")

    try:
        t0 = time.perf_counter()
        with torch.no_grad():
            out = model_hf.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                use_cache=True,        # keep KV cache on; important for TPS measurement
            )
        elapsed = time.perf_counter() - t0
    except torch.cuda.OutOfMemoryError:
        print(f"CUDA OOM during inference — reduce MAX_NEW_TOKENS (currently {MAX_NEW_TOKENS})")
        raise

    new_tokens = out.shape[1] - inputs["input_ids"].shape[1]
    tps        = new_tokens / elapsed
    print(f"\nGenerated {new_tokens} tokens in {elapsed:.2f}s  →  {tps:.1f} TPS  (naive HF baseline)")
    print(tokenizer.decode(out[0], skip_special_tokens=True))

    # Post-inference VRAM (KV cache is retained until del model_hf)
    print(f"\nVRAM after inference: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

### 6a. TODO: Write `QwenMoEAdapter`

Once the model loads, the next step is bridging it to Dana's internal interfaces:

```python
# engine/dana-engine/src/dana_engine/model/qwen_adapter.py  (to be created)
class QwenMoEAdapter:
    """Wraps a HuggingFace Qwen3.5-MoE model with Dana engine interfaces.

    Qwen3.5-35B-A3B architecture notes:
      - 256 experts per MoE layer (vs 60 in Qwen1.5-MoE)
      - Hybrid: Gated Delta Network layers interleaved with standard attention
      - Expert FFN: gate_proj + up_proj + down_proj  (hidden=2048, intermediate≈1024)
      - Routing: top-K sparse, K varies by layer config

    Maps:
      model.model.layers[i].mlp.experts[j].gate_proj.weight  →  TieredTensorStore
      model.model.layers[i].mlp.gate.weight                   →  RouterPredictor
    """
    def __init__(self, hf_model, store: TieredTensorStore):
        self.model = hf_model
        self.store = store
        self._offload_experts()         # move all experts to RAM tier

    def _offload_experts(self):
        for layer_idx, layer in enumerate(self.model.model.layers):
            if hasattr(layer.mlp, 'experts'):
                for exp_idx, expert in enumerate(layer.mlp.experts):
                    for proj_name in ('gate_proj', 'up_proj', 'down_proj'):
                        proj = getattr(expert, proj_name, None)
                        if proj is not None:
                            key = f"L{layer_idx}_E{exp_idx}_{proj_name}"
                            self.store.store(key, proj.weight.cpu(), tier="ram")
                            proj.weight = None   # free VRAM
```

**Status: TODO (Priority 1a in NEXT_STEPS.md)**

---
## 7. GPU TODO 4 — Quantization Speed Benchmark <a id="7-quant"></a>

Compare Q2/Q4/Q8 dequantization throughput on GPU.

In [ ]:
import torch, time, statistics
from moe_quant import quantize, dequantize

if not torch.cuda.is_available():
    print("No GPU — running CPU dequant benchmark instead")
    device = "cpu"
else:
    device = "cuda"

expert_weight = torch.randn(4096, 4096)
if device == "cuda":
    expert_weight = expert_weight.cuda()

print(f"Quantization benchmark — device: {device}")
print(f"  Expert shape: {expert_weight.shape}  ({expert_weight.element_size() * expert_weight.numel() / 1024**2:.0f} MB FP32)\n")
print(f"  {'Bits':<6} {'Dequant µs':>12} {'Compression':>12} {'Memory MB':>10}")
print("  " + "-" * 44)

for bits in [2, 4, 8]:
    qt = quantize(expert_weight.cpu(), bits=bits)
    compression = expert_weight.element_size() * expert_weight.numel() / qt.nbytes
    mem_mb = qt.nbytes / 1024**2

    N = 500 if device == "cuda" else 50
    times = []
    for _ in range(N):
        t0 = time.perf_counter()
        dequantize(qt)
        if device == "cuda":
            torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)

    avg_us = statistics.median(times) * 1e6
    print(f"  Q{bits:<5} {avg_us:>12.1f} {compression:>12.1f}x {mem_mb:>10.1f}")

---
## 8. GPU TODO 5 — Baseline vs Full Pipeline Benchmark <a id="8-pipeline"></a>

Compare TPS at each optimization stage using a real Qwen model.

**Prerequisite:** Section 6 (Qwen model downloaded and loaded).

**Expected TPS progression on 2× RTX 4090 + 512 GB DDR5:**
```
naive          ~0.5 TPS
+prefetch      ~2.5 TPS
+cache         ~6.0 TPS
+quant         ~15  TPS
+spec          ~40-60 TPS
```

In [ ]:
import torch, time

if not torch.cuda.is_available():
    print("No GPU — skipping real pipeline benchmark")
else:
    try:
        model_hf  # must have been loaded in Section 6
    except NameError:
        print("Load the Qwen model in Section 6 first.")
    else:
        # This benchmark requires QwenMoEAdapter (TODO Priority 1a).
        # For now we show the interface and measure naive HF baseline.
        prompt = "Dana MoE engine is"
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        MAX_NEW = 100

        stages = [
            ("naive (HuggingFace)",  dict(do_sample=False)),
        ]

        print(f"{'Stage':<30} {'TPS':>8}")
        print("-" * 42)

        for label, kwargs in stages:
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            with torch.no_grad():
                out = model_hf.generate(**inputs, max_new_tokens=MAX_NEW, **kwargs)
            torch.cuda.synchronize()
            elapsed = time.perf_counter() - t0
            new_tokens = out.shape[1] - inputs["input_ids"].shape[1]
            tps = new_tokens / elapsed
            print(f"{label:<30} {tps:>8.1f}")

        print("\n(Dana pillars benchmark requires QwenMoEAdapter — see Section 6a TODO)")

---
## 9. GPU TODO 6 — Expert Activation Heatmap <a id="9-heatmap"></a>

Visualize which of Qwen3.5-35B-A3B's 256 experts are "hot" across Persian and English prompts.  
This validates the power-law assumption that justifies `expert-cache` thresholds.

**Architecture note:** Qwen3.5 uses a hybrid layout — some layers are Gated Delta Network (linear attention, no MoE) and some are standard attention with MoE FFN. The hook below targets only MoE layers by checking `hasattr(layer.mlp, 'gate')`.

In [ ]:
import torch, collections

if not torch.cuda.is_available():
    print("No GPU — skipping heatmap")
else:
    try:
        model_hf
    except NameError:
        print("Load Qwen model first (Section 6).")
    else:
        prompts = [
            ("FA", "سلام، چطوری؟"),
            ("EN", "Hello, how are you?"),
            ("FA", "برنامه‌نویسی پایتون چیست؟"),
            ("EN", "What is Python programming?"),
            ("FA", "مدل‌های زبانی بزرگ چگونه کار می‌کنند؟"),
            ("EN", "How do large language models work?"),
        ]

        expert_counts: dict[str, dict[int, int]] = {"FA": {}, "EN": {}}
        hooks = []

        def make_hook(layer_idx, lang_ref):
            def hook(module, input, output):
                # Qwen3.5 Qwen3_5MoeSparseMoeBlock returns (hidden, router_logits)
                # router_logits shape: (batch * seq, num_experts) or (batch, seq, num_experts)
                if isinstance(output, tuple) and len(output) >= 2:
                    router_logits = output[1]
                    if router_logits.dim() == 2:
                        # (batch*seq, num_experts) — standard sparse MoE format
                        top_ids = router_logits.topk(min(8, router_logits.shape[-1]), dim=-1).indices
                    else:
                        top_ids = router_logits.topk(min(8, router_logits.shape[-1]), dim=-1).indices
                    d = expert_counts[lang_ref[0]]
                    for eid in top_ids.flatten().cpu().tolist():
                        d[eid] = d.get(eid, 0) + 1
            return hook

        for lang, prompt in prompts:
            lang_ref = [lang]  # mutable container for closure
            for i, layer in enumerate(model_hf.model.layers):
                if hasattr(layer, 'mlp') and hasattr(layer.mlp, 'gate'):
                    h = layer.mlp.register_forward_hook(make_hook(i, lang_ref))
                    hooks.append(h)

            inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
            with torch.no_grad():
                model_hf(**inputs)

            for h in hooks:
                h.remove()
            hooks.clear()

        print("Top 10 hot experts by language (expert ID → activation count):")
        for lang in ["FA", "EN"]:
            sorted_experts = sorted(expert_counts[lang].items(), key=lambda x: -x[1])[:10]
            print(f"  {lang}: {sorted_experts}")

        # Overlap between FA and EN top-20
        fa_top20 = {e for e, _ in sorted(expert_counts["FA"].items(), key=lambda x: -x[1])[:20]}
        en_top20 = {e for e, _ in sorted(expert_counts["EN"].items(), key=lambda x: -x[1])[:20]}
        overlap = fa_top20 & en_top20
        total_experts = 256  # Qwen3.5-35B-A3B has 256 experts per layer
        print(f"\nFA∩EN top-20 overlap: {len(overlap)}/{total_experts} experts — {overlap}")
        print("(High overlap → shared hot tier. Low overlap → language-specific caching needed.)")
        print(f"Power-law check: top-20 experts = {20/total_experts:.0%} of pool — should serve ~80%+ of activations")

---
## 10. GPU TODO 7 — CUDA Kernel: Tree Verification <a id="10-cuda"></a>

The reference `spec-decode-tree` uses pure Python. For production speed, a CUDA kernel
is needed for parallel tree path verification.

**Reference algorithm:** `spec-decode-tree/src/spec_decode_tree/verify.py`  
**Target file:** `dana-engine/csrc/tree_verify.cu`

### Quick win (no CUDA required): batch tree path verification in Python

In [ ]:
import torch, time
from dana_engine.model.config import TinyMoEConfig
from dana_engine.model.transformer import TinyMoETransformer

cfg   = TinyMoEConfig.micro()
model = TinyMoETransformer(cfg).eval()

# Simulate tree paths: (num_paths, context_len)
context     = torch.randint(0, cfg.vocab_size, (1, 8))
tree_paths  = [torch.randint(0, cfg.vocab_size, (1, i+1)) for i in range(4)]  # depth-4 tree

# --- CURRENT (one forward per path) ---
t0 = time.perf_counter()
for _ in range(20):
    for path in tree_paths:
        full = torch.cat([context, path], dim=1)
        logits = model(full).logits
naive_ms = (time.perf_counter() - t0) / 20 * 1000

# --- BATCHED (one forward for all paths) ---
max_len = max(torch.cat([context, p], dim=1).shape[1] for p in tree_paths)
padded  = []
for path in tree_paths:
    full = torch.cat([context, path], dim=1)
    # pad to max_len
    pad = max_len - full.shape[1]
    if pad > 0:
        full = torch.cat([full, torch.zeros(1, pad, dtype=torch.long)], dim=1)
    padded.append(full)

batch = torch.cat(padded, dim=0)   # (num_paths, max_len)

t0 = time.perf_counter()
for _ in range(20):
    logits_batch = model(batch).logits  # (num_paths, max_len, vocab)
batched_ms = (time.perf_counter() - t0) / 20 * 1000

print(f"Naïve (one fwd/path) : {naive_ms:.2f} ms")
print(f"Batched (all paths)  : {batched_ms:.2f} ms")
print(f"Speedup              : {naive_ms / batched_ms:.2f}x")
print("\nThis batched approach is the 'Quick Win' from NEXT_STEPS.md")

### 10b. CUDA kernel stub

The skeleton for the CUDA tree verification kernel lives at:  
`dana-engine/csrc/tree_verify.cu` *(to be created)*

```c
// tree_verify.cu — parallel tree path verification
//
// Input:  logits     (num_paths, tree_size, vocab_size)  float16
//         candidates (num_paths, tree_size)              int32  draft token ids
//         target_ids (num_paths, tree_size)              int32  target greedy ids
// Output: accept_mask (num_paths, tree_size)             bool
//         best_path   (num_paths,)                       int32  index of longest valid path
//
// Each thread block handles one tree path.
// Each thread handles one node in the path.

__global__ void tree_verify_kernel(
    const __half* logits,
    const int*    candidates,
    const int*    target_ids,
    bool*         accept_mask,
    int*          best_path,
    int           num_paths,
    int           tree_size,
    int           vocab_size
) {
    int path_idx = blockIdx.x;
    int node_idx = threadIdx.x;
    // ...
}
```

**Status: TODO (Priority 7 in GPU_TODO.md)**

---
## 11. GPU TODO 8 — Production API Stress Test <a id="11-stress"></a>

**Target:** 20–30 TPS/user at 10 concurrent users on 2× RTX 4090 + 512 GB DDR5.

In [ ]:
# Step 1: Start the Dana API server in the background
import subprocess, time, os

SERVER_PORT = 8765  # use non-standard port to avoid conflicts

server_proc = subprocess.Popen(
    ["python", "-m", "dana_engine.api.server", "--port", str(SERVER_PORT)],
    cwd=ENGINE_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
time.sleep(2.0)   # let server start

# Ping health endpoint
import urllib.request, json
try:
    resp = urllib.request.urlopen(f"http://localhost:{SERVER_PORT}/health", timeout=3)
    health = json.loads(resp.read())
    print("Server up:", health)
except Exception as e:
    print(f"Server not ready: {e} — check the server process")

In [ ]:
# Step 2: Locust stress test
!pip install locust --quiet

locust_file = """
from locust import HttpUser, task, between
import random

PROMPTS = [
    "سلام", "Hello", "برنامه‌نویسی", "Python", "هوش مصنوعی", "AI inference",
]

class DanaUser(HttpUser):
    wait_time = between(0.5, 1.5)

    @task
    def chat(self):
        self.client.post("/v1/chat/completions", json={
            "model": "dana-tiny",
            "messages": [{"role": "user", "content": random.choice(PROMPTS)}],
            "max_tokens": 32,
        }, timeout=30)
"""

with open("/tmp/locustfile.py", "w") as f:
    f.write(locust_file)

print("Locust file written to /tmp/locustfile.py")
print(f"Run: locust -f /tmp/locustfile.py --host=http://localhost:{SERVER_PORT} --users 10 --spawn-rate 2 --run-time 30s --headless")

In [ ]:
!locust -f /tmp/locustfile.py \
    --host=http://localhost:{SERVER_PORT} \
    --users 10 --spawn-rate 2 \
    --run-time 30s --headless \
    --csv=/tmp/locust_results 2>&1 | tail -30

In [ ]:
# Clean up server process
try:
    server_proc.terminate()
    server_proc.wait(timeout=5)
    print("Server stopped.")
except Exception:
    pass

---
## 12. GPU TODO 9 — GPU Memory Profiling <a id="12-mem"></a>

Capture PyTorch memory snapshots to identify VRAM leaks and over-allocation.

In [ ]:
import torch

if not torch.cuda.is_available():
    print("No GPU — skipping memory profiling")
else:
    from dana_engine.model.config import TinyMoEConfig
    from dana_engine.model.transformer import TinyMoETransformer
    from dana_engine.pipeline import DanaInferencePipeline, PipelineConfig

    cfg = TinyMoEConfig.tiny()   # larger than micro for meaningful VRAM footprint
    pipeline = DanaInferencePipeline(PipelineConfig(
        model_config=cfg,
        enable_spec_decode=True,
        num_draft_tokens=4,
        enable_prefetch=True,
        max_new_tokens=50,
    ))

    # Move model to GPU
    pipeline._model = pipeline._model.cuda()

    torch.cuda.reset_peak_memory_stats()
    torch.cuda.memory._record_memory_history(max_entries=100_000)

    input_ids = torch.randint(0, cfg.vocab_size, (1, 16)).cuda()
    result = pipeline.generate(input_ids, max_new_tokens=50)

    peak_mb = torch.cuda.max_memory_allocated() / 1024**2
    curr_mb = torch.cuda.memory_allocated()     / 1024**2

    # Save snapshot for Chrome trace
    snapshot_path = "/tmp/dana_memory_snapshot.pickle"
    torch.cuda.memory._dump_snapshot(snapshot_path)
    torch.cuda.memory._record_memory_history(enabled=None)  # stop recording

    print(f"Peak VRAM : {peak_mb:.1f} MB")
    print(f"Current   : {curr_mb:.1f} MB")
    print(f"Generated : {result.tokens_generated} tokens at {result.tokens_per_second:.1f} TPS")
    print(f"Snapshot  : {snapshot_path}")
    print("\nView in: python -m torch.cuda.memory --snapshot", snapshot_path)

---
## 13. Next Steps Tracker <a id="13-next-steps"></a>

Edit the dict below then run the cell — progress is saved to `/content/dana/dana_todo.json`.  
Note: this file lives on Colab's local disk and is lost when the session ends. Copy the output manually if you want to preserve it.

In [ ]:
import json, pathlib

TODO_PATH = DRIVE_ROOT / "dana_todo.json"

# Load saved progress if it exists in this session
if TODO_PATH.exists():
    with open(TODO_PATH) as f:
        TODO_TRACKER = json.load(f)
    print(f"Loaded saved progress from {TODO_PATH}")
else:
    # Default state — all TODO
    TODO_TRACKER = {
        # Priority 1 — Validate core claims on real hardware
        "P1a Write QwenMoEAdapter (HF model → Dana interfaces)": "TODO",
        "P1b Measure real PCIe bandwidth per expert size": "TODO",
        "P1c Measure naïve vs prefetch TPS on real model": "TODO",
        # Priority 2 — Fix CPU/GPU tier gap
        "P2a Wire CUDA tensors into tiered-tensor-store hot tier": "TODO",
        "P2b Implement real async H2D in AsyncExpertLoader": "TODO",
        "P2c CUDA/Triton kernel for Q4/Q2 dequantization on GPU": "TODO",
        # Priority 3 — Spec decode tuning
        "P3a Measure acceptance rate on real model (OpenHermes)": "TODO",
        "P3b Cap tree spec decode depth/width with budget limit (max_paths=32)": "TODO",
        "P3c Evaluate draft model alternatives (layer-skip, Medusa)": "TODO",
        # Priority 4 — Expert cache tuning
        "P4a Profile real expert access frequency (power-law fit)": "TODO",
        "P4b Validate Jaccard batching assumption on real prompts": "TODO",
        # Priority 5 — Production hardening
        "P5a Replace synthetic tokenizer with AutoTokenizer": "TODO",
        "P5b Wire ExpertAwareBatchScheduler into pipeline (concurrent requests)": "TODO",
        "P5c Connect VRAMBudgetManager to pipeline eviction loop": "TODO",
        # Priority 6 — Colab end-to-end validation
        "P6  Run full Colab smoke test (Steps 1–9 in NEXT_STEPS.md)": "TODO",
        # Quick wins
        "QW  Batch tree path verification (one fwd for all paths)": "TODO",
        "QW  CUDA kernel for tree verification (tree_verify.cu)": "TODO",
    }

# ── Edit statuses here ─────────────────────────────────────────────────
# Values: "TODO" | "IN_PROGRESS" | "DONE" | "BLOCKED"
# Example: TODO_TRACKER["P1b Measure real PCIe bandwidth per expert size"] = "DONE"
# ──────────────────────────────────────────────────────────────────────

with open(TODO_PATH, "w") as f:
    json.dump(TODO_TRACKER, f, indent=2)
print(f"Saved to {TODO_PATH}  (local session only — copy output to preserve across sessions)
")

# Print status table
STATUS_ICONS = {"TODO": "○", "IN_PROGRESS": "◐", "DONE": "●", "BLOCKED": "✗"}
print("DANA ENGINE — TODO STATUS")
print("=" * 70)
for task, status in TODO_TRACKER.items():
    icon = STATUS_ICONS.get(status, "?")
    print(f"  {icon} [{status:<12}] {task}")
print("=" * 70)

done  = sum(1 for s in TODO_TRACKER.values() if s == "DONE")
total = len(TODO_TRACKER)
pct   = 100 * done // total if total else 0
print(f"
Progress: {done}/{total} done  ({pct}%)")

---
## 14. Known Bugs & Quick Wins <a id="14-bugs"></a>

In [ ]:
KNOWN_BUGS = [
    {
        "id": "BUG-1",
        "title": "Tree verifier runs one forward per path (should batch)",
        "location": "spec-decode-tree/src/spec_decode_tree/verify.py",
        "impact": "HIGH — tree TPS collapses on CPU",
        "fix": "Batch all paths into one forward pass (see Section 10)",
        "status": "OPEN",
    },
    {
        "id": "BUG-2",
        "title": "_prefetch_from_draft() is a no-op",
        "location": "dana-engine/src/dana_engine/pipeline.py:253",
        "impact": "MEDIUM — prefetch has no effect yet",
        "fix": "Wire to AsyncExpertLoader.enqueue() with CUDA streams",
        "status": "OPEN",
    },
    {
        "id": "BUG-3",
        "title": "VRAMBudgetManager not connected to eviction loop",
        "location": "dana-engine/src/dana_engine/pipeline.py",
        "impact": "MEDIUM — unbounded VRAM growth on long sessions",
        "fix": "Call enforce() before each expert load in pipeline loop",
        "status": "OPEN",
    },
    {
        "id": "BUG-4",
        "title": "Synthetic tokenizer produces garbage text output",
        "location": "dana-engine/src/dana_engine/api/server.py",
        "impact": "LOW — only affects test model output readability",
        "fix": "Swap in AutoTokenizer from transformers",
        "status": "OPEN",
    },
    {
        "id": "BUG-5",
        "title": "AsyncExpertLoader double-buffer logic is CPU-only",
        "location": "moe-router-predict/src/moe_router_predict/async_loader.py",
        "impact": "HIGH — no real async on GPU",
        "fix": "Use torch.cuda.Stream() + non_blocking=True",
        "status": "OPEN",
    },
]

print("KNOWN BUGS")
print("=" * 70)
for bug in KNOWN_BUGS:
    print(f"  [{bug['id']}] {bug['title']}")
    print(f"         Location : {bug['location']}")
    print(f"         Impact   : {bug['impact']}")
    print(f"         Fix      : {bug['fix']}")
    print(f"         Status   : {bug['status']}")
    print()

### 14a. Quick Win: Wire tree verifier batching (5-min fix)

The single most impactful code change that requires no GPU:

In [ ]:
# Show the patch for the tree verifier batching fix.
# Apply this to spec-decode-tree/src/spec_decode_tree/verify.py

PATCH = '''
# BEFORE (in verify.py) — one forward pass per candidate path:
verified_tokens = []
for path in tree_paths:
    full_ids = torch.cat([context_ids, path], dim=1)
    logits   = model(full_ids).logits          # (1, seq, vocab)
    target   = logits[0, -len(path):, :].argmax(-1)
    accept   = (target == path[0]).all().item()
    if accept:
        verified_tokens = path[0].tolist()

# AFTER — one batched forward for all paths:
max_len  = max(context_ids.shape[1] + len(p[0]) for p in tree_paths)
padded   = []
for path in tree_paths:
    full = torch.cat([context_ids, path], dim=1)
    pad  = max_len - full.shape[1]
    if pad > 0:
        full = torch.cat([full, torch.zeros(1, pad, dtype=torch.long)], dim=1)
    padded.append(full)

batch      = torch.cat(padded, dim=0)           # (num_paths, max_len)
all_logits = model(batch).logits               # (num_paths, max_len, vocab)

verified_tokens = []
for i, path in enumerate(tree_paths):
    path_len = len(path[0])
    path_start = context_ids.shape[1]
    target = all_logits[i, path_start:path_start+path_len, :].argmax(-1)
    if (target == path[0]).all():
        verified_tokens = path[0].tolist()
        break   # longest accepted path wins
'''

print(PATCH)

---
## Contributing

**Branch:** `claude/plan-dana-engine-HC5zg`  
**Dev workflow:**

```bash
git checkout claude/plan-dana-engine-HC5zg
git pull origin claude/plan-dana-engine-HC5zg

# Make changes, then:
cd engine && python -m pytest dana-engine/tests/ -v   # run all tests
git add -p && git commit -m "feat: your change"
git push -u origin claude/plan-dana-engine-HC5zg
```

**Priority order for next contributor:**
1. Fix `BUG-1` (tree verifier batching) — 5 min, huge win
2. `P2a` wire `.cuda()` into `tier_manager.py` hot tier
3. `P2b` implement real async H2D in `AsyncExpertLoader`
4. `P1a` write `QwenMoEAdapter`
5. Run Colab end-to-end smoke test (`P6`)
